# Check Metrics

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import sys

sys.path.append("..")
from src.utils import load_json

data_name = 'cifar10'
optim = 'sgd'

data = load_json(f"../result/target_linearity/{data_name}_{optim}.json")

# Train and Test Accuracy
train_accuracy = np.array([0.0] + data['train_accuracy'])
test_accuracy = np.array([0.0] + data['test_accuracy'])

# Surrogate and Target Linearity
surrogate = np.array(data['train_surrogate']).T
norm = np.array(data['train_norms']).T

target_linearity = np.array(data['train_target_linearity']).T
target_linearity = np.clip(target_linearity, min=0.0)


In [ ]:
n_layers = surrogate.shape[0]
fig, axes = plt.subplots(ncols=2, nrows=1, figsize=(9, 4)) 
cmap = cm.viridis
colors = [cmap(1 - (i / (n_layers - 1))) for i in range(n_layers)]

# data_lin[0] will be surrogate, data_lin[1] will be target_linearity
data_lin = [surrogate, target_linearity] 
y_labels = ['Surrogate', 'Target Linearity']

for i, ax in enumerate(axes):
    # --- Create Twin Axis for Accuracy ---
    ax_acc = ax.twinx()
    
    # Plot Accuracy (Range 0-1)
    ax_acc.plot(train_accuracy, color='blue', linestyle='--', alpha=0.3, label='Train Acc', linewidth=6)
    ax_acc.plot(test_accuracy, color='red', linestyle='--', alpha=0.3, label='Test Acc', linewidth=6)
    ax_acc.set_ylim(-0.05, 1.05) # Keep accuracy bounded
    
    # --- Plot Linearity Data ---
    for layer_idx in range(n_layers):
        ax.plot(data_lin[i][layer_idx], 
                color=colors[layer_idx], 
                linewidth=6, 
                label=f'Layer {layer_idx+1}')

    # Formatting
    ax.set_title(y_labels[i], weight='bold', size=28)
    ax.grid(True, which='both', linestyle=':', alpha=0.5)
    
    if i == 1:
        ax_acc.set_ylabel('Accuracy', weight='bold', size=28)
        
    # Combined Legend for the second plot
    if i == 1:
        lines, labels = ax.get_legend_handles_labels()
        lines2, labels2 = ax_acc.get_legend_handles_labels()
        # ax.legend(lines + lines2, labels + labels2, loc='lower right', fontsize='medium', prop={'weight': 'bold'})

# fig.suptitle(f'{data_name.upper()}', weight='bold', size=20)
fig.supxlabel(data_name.upper(), weight='bold', size=28)
plt.tight_layout()
plt.show()